# Drawing Modality — Spiral Dataset (Otsu + HOG + LBP + comparaison classifieurs)

Ce notebook entraîne un modèle de détection de la maladie de Parkinson à partir du
**dataset Spiral** (`data/spiral`), qui est structurellement différent du dataset NewHandPD :

| | NewHandPD (`data/handpd`) | Spiral (`data/spiral`) |
|---|---|---|
| Format | JPG | PNG |
| Split | manuel (20 % sujets) | pré-défini par auteurs |
| Train | ~80 % | 36 HC + 36 PD |
| Test | ~20 % | 15 HC + 15 PD |


## 0. Dépendances

In [1]:
import subprocess, sys

packages = [
    "numpy", "pandas", "scikit-learn", "scikit-image",
    "Pillow", "matplotlib", "scipy", "joblib",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("Tous les packages sont prêts.")

Tous les packages sont prêts.


## 1. Chemins et hyper-paramètres

In [2]:
from pathlib import Path
import numpy as np

REPO_ROOT   = Path('.').resolve().parent.parent  
DATA_ROOT   = REPO_ROOT / 'data' / 'spiral'
MODEL_DIR   = REPO_ROOT / 'models'
MODEL_PATH  = MODEL_DIR / 'drawing_spiral_v2_hog_lbp_pipeline.joblib'

IMG_SIZE         = (128, 128)    
HOG_ORIENTATIONS = 9
HOG_PIXELS_CELL  = (8, 8)       
HOG_CELLS_BLOCK  = (2, 2)
LBP_RADIUS       = 3            
LBP_N_POINTS     = 8 * LBP_RADIUS  
LBP_N_BINS       = 26          

RANDOM_STATE = 42

for split in ('training', 'testing'):
    for cls in ('healthy', 'parkinson'):
        folder = DATA_ROOT / split / cls
        n = len(list(folder.glob('*.png')))
        print(f"{split}/{cls:>10} : {n} images")
print(f"\nREPO_ROOT : {REPO_ROOT}")

training/   healthy : 36 images
training/ parkinson : 36 images
testing/   healthy : 15 images
testing/ parkinson : 15 images

REPO_ROOT : C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection


## 2. Chargement du dataset

Le split train/test est **pré-défini** par les auteurs du dataset.
Nous respectons cette séparation et n'appliquons l'augmentation **qu'au train**.

In [3]:
from PIL import Image


def load_split(split: str):
    """Charge toutes les images d'un split ('training' ou 'testing').
    
    Retourne (images_PIL, labels, subject_ids).
    Nom de fichier ex : V01HE02.png  →  sujet='V01', classe='HC' (H=healthy)
    """
    imgs, labs, sids = [], [], []
    for label, cls in enumerate(('healthy', 'parkinson')):
        folder = DATA_ROOT / split / cls
        for path in sorted(folder.glob('*.png')):
            sid = path.stem[:3]
            imgs.append(Image.open(path).convert('L'))
            labs.append(label)   # 0=HC, 1=PD
            sids.append(sid)
    return imgs, labs, sids


train_imgs, train_labs, train_sids = load_split('training')
test_imgs,  test_labs,  test_sids  = load_split('testing')

print(f"Train : {len(train_imgs)} images  ({sum(l==0 for l in train_labs)} HC / {sum(l==1 for l in train_labs)} PD)")
print(f"Test  : {len(test_imgs)} images  ({sum(l==0 for l in test_labs)} HC / {sum(l==1 for l in test_labs)} PD)")
print(f"Sujets train uniques : {sorted(set(train_sids))}")

Train : 72 images  (36 HC / 36 PD)
Test  : 30 images  (15 HC / 15 PD)
Sujets train uniques : ['V01', 'V02', 'V03', 'V04', 'V05', 'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V55']


## 3. Visualisation du dataset

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

hc_idx = [i for i, l in enumerate(train_labs) if l == 0]
pd_idx = [i for i, l in enumerate(train_labs) if l == 1]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for col, idx in enumerate(hc_idx[:5]):
    axes[0, col].imshow(train_imgs[idx], cmap='gray')
    axes[0, col].set_title(f'HC — {train_sids[idx]}', fontsize=9)
    axes[0, col].axis('off')
for col, idx in enumerate(pd_idx[:5]):
    axes[1, col].imshow(train_imgs[idx], cmap='gray')
    axes[1, col].set_title(f'PD — {train_sids[idx]}', fontsize=9)
    axes[1, col].axis('off')

fig.suptitle('Exemples du train (haut : HC, bas : PD)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_ROOT / 'samples_train.png', dpi=80)
plt.show()
print('Figure sauvegardée.')

Figure sauvegardée.


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_32580\2396519209.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Augmentation ×8 — géométrique + texture synthétique (train uniquement)

Le dataset train est très petit (72 images). On applique **8 transformations** par image :

- 4 rotations (0°, 90°, 180°, 270°) — invariance à l'orientation
- flip horizontal + flip vertical — invariance à la main dominante
- **flou gaussien léger** (radius=0.8) — simule un trait canvas fluide (anti-aliasing numérique)
- **filtre de netteté** — simule un trait papier bien contrasté (encre sur fibre)



In [5]:
from PIL import ImageOps, ImageFilter

AUGMENTATIONS = [
    ('rot0',    lambda img: img),
    ('rot90',   lambda img: img.rotate(90, expand=False)),
    ('rot180',  lambda img: img.rotate(180, expand=False)),
    ('rot270',  lambda img: img.rotate(270, expand=False)),
    ('flipH',   lambda img: ImageOps.mirror(img)),
    ('flipV',   lambda img: ImageOps.flip(img)),
    ('blur',    lambda img: img.filter(ImageFilter.GaussianBlur(radius=0.8))),
    ('sharpen', lambda img: img.filter(ImageFilter.SHARPEN)),
]

aug_imgs, aug_labs, aug_sids = [], [], []
for img, lab, sid in zip(train_imgs, train_labs, train_sids):
    for aug_name, aug_fn in AUGMENTATIONS:
        aug_imgs.append(aug_fn(img))
        aug_labs.append(lab)
        aug_sids.append(sid)

print(f"Après augmentation : {len(aug_imgs)} images train  (×{len(AUGMENTATIONS)} par image)")
print(f"  HC : {sum(l==0 for l in aug_labs)}  |  PD : {sum(l==1 for l in aug_labs)}")


Après augmentation : 576 images train  (×8 par image)
  HC : 288  |  PD : 288


## 5. Prétraitement et extraction de features

### Binarisation d'Otsu (correction domain shift)


In [6]:
from skimage.feature import hog, local_binary_pattern
from skimage.filters import threshold_otsu


def preprocess_image(pil_img: Image.Image, target_size=IMG_SIZE) -> np.ndarray:
    img = pil_img.convert('L').resize(target_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    thresh = threshold_otsu(arr)           # seuil optimal par image
    return (arr < thresh).astype(np.float32) 


def extract_hog_features(arr: np.ndarray) -> np.ndarray:
    """Vecteur HOG 1-D sur image binarisée."""
    return hog(
        arr,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_CELL,
        cells_per_block=HOG_CELLS_BLOCK,
        block_norm='L2-Hys',
        feature_vector=True,
    ).astype(np.float32)


def extract_lbp_features(arr: np.ndarray) -> np.ndarray:
    lbp = local_binary_pattern(
        arr,
        P=LBP_N_POINTS,
        R=LBP_RADIUS,
        method='uniform',
    )
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=LBP_N_BINS,
        range=(0, LBP_N_BINS),
    )
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist


def extract_features(pil_img: Image.Image) -> np.ndarray:
    """Pipeline complet : PIL Image → vecteur HOG+LBP sur image binarisée."""
    arr = preprocess_image(pil_img)
    return np.concatenate([extract_hog_features(arr), extract_lbp_features(arr)])


X_train = np.array([extract_features(img) for img in aug_imgs])
y_train = np.array(aug_labs)
g_train = np.array(aug_sids)

X_test = np.array([extract_features(img) for img in test_imgs])
y_test = np.array(test_labs)

hog_dim = extract_hog_features(preprocess_image(train_imgs[0])).shape[0]
print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"HOG : {hog_dim} features  |  LBP : {LBP_N_BINS} features  |  Total : {X_train.shape[1]}")


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


X_train : (576, 8126)  |  X_test : (30, 8126)
HOG : 8100 features  |  LBP : 26 features  |  Total : 8126


## 6. Comparaison de classifieurs par validation croisée

On utilise `StratifiedGroupKFold` pour que les 6 augmentations d'un même sujet
restent **toujours dans le même pli** (pas de fuite de données).

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import roc_auc_score
import pandas as pd

unique_subjects = np.unique(g_train)
n_splits = min(5, len(unique_subjects) // 2)
cv = StratifiedGroupKFold(n_splits=n_splits)
print(f"Validation croisée : {n_splits} plis, {len(unique_subjects)} sujets uniques")

CLASSIFIERS = {
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=10.0, gamma='scale',
                    probability=True, random_state=RANDOM_STATE)),
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=300, max_depth=None,
            min_samples_leaf=2, random_state=RANDOM_STATE,
        )),
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=300, max_depth=3,
            learning_rate=0.05, subsample=0.8,
            random_state=RANDOM_STATE,
        )),
    ]),
}

results = {}
for name, pipe in CLASSIFIERS.items():
    cv_res = cross_validate(
        pipe, X_train, y_train,
        groups=g_train,
        cv=cv,
        scoring='roc_auc',
        return_train_score=False,
    )
    aucs = cv_res['test_score']
    results[name] = {'mean': aucs.mean(), 'std': aucs.std(), 'scores': aucs}
    print(f"{name:25s}  AUC = {aucs.mean():.3f} ± {aucs.std():.3f}")

best_clf_name = max(results, key=lambda k: results[k]['mean'])
print(f"\n>>> Meilleur classifieur : {best_clf_name}  (AUC={results[best_clf_name]['mean']:.3f})")

Validation croisée : 5 plis, 16 sujets uniques
SVM (RBF)                  AUC = 0.854 ± 0.053
Random Forest              AUC = 0.871 ± 0.030
Gradient Boosting          AUC = 0.864 ± 0.124

>>> Meilleur classifieur : Random Forest  (AUC=0.871)


## 7. Visualisation de la comparaison

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

names  = list(results.keys())
means  = [results[n]['mean'] for n in names]
stds   = [results[n]['std']  for n in names]
colors = ['steelblue' if n != best_clf_name else 'darkorange' for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, means, yerr=stds, capsize=6, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('AUC (CV)', fontsize=11)
ax.set_title('Comparaison des classifieurs — dataset Spiral', fontsize=12, fontweight='bold')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.01,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=10)
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='Aléatoire')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_ROOT / 'classifier_comparison.png', dpi=80)
plt.show()
print('Figure sauvegardée.')

Figure sauvegardée.


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_32580\4072303456.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Recherche du seuil de décision optimal (OOF)

On calcule les prédictions **out-of-fold** avec le meilleur classifieur
pour trouver le seuil qui maximise le F1-score.

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, f1_score
from sklearn.base import clone

best_pipe = CLASSIFIERS[best_clf_name]

y_oof = np.zeros(len(y_train))
for tr_idx, val_idx in cv.split(X_train, y_train, g_train):
    fold_pipe = clone(best_pipe)
    fold_pipe.fit(X_train[tr_idx], y_train[tr_idx])
    y_oof[val_idx] = fold_pipe.predict_proba(X_train[val_idx])[:, 1]

fpr, tpr, thresholds = roc_curve(y_train, y_oof)
f1_scores_thresh = [f1_score(y_train, y_oof >= t, zero_division=0) for t in thresholds]
best_idx = int(np.argmax(f1_scores_thresh))
DECISION_THRESHOLD = float(thresholds[best_idx])

oof_auc = roc_auc_score(y_train, y_oof)
print(f"AUC OOF global   : {oof_auc:.3f}")
print(f"Seuil F1-optimal : {DECISION_THRESHOLD:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(fpr, tpr, color='steelblue', label=f'AUC={oof_auc:.3f}')
ax1.plot([0, 1], [0, 1], '--', color='gray')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('Courbe ROC — OOF'); ax1.legend()
ax2.plot(thresholds, f1_scores_thresh, color='darkorange')
ax2.axvline(DECISION_THRESHOLD, color='red', linestyle='--',
            label=f'seuil={DECISION_THRESHOLD:.2f}')
ax2.set_xlabel('Seuil'); ax2.set_ylabel('F1')
ax2.set_title('F1 vs seuil (OOF)'); ax2.legend()
plt.tight_layout()
plt.savefig(DATA_ROOT / 'roc_threshold_spiral.png', dpi=80)
plt.show()

AUC OOF global   : 0.788
Seuil F1-optimal : 0.499


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_32580\1109378321.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Entraînement final sur tout le train augmenté

In [10]:
from sklearn.base import clone

final_pipeline = clone(best_pipe)
final_pipeline.fit(X_train, y_train)

train_proba = final_pipeline.predict_proba(X_train)[:, 1]
train_auc   = roc_auc_score(y_train, train_proba)

print(f"Classifieur retenu : {best_clf_name}")
print(f"AUC train (full)   : {train_auc:.3f}")
print(f"AUC CV (mean±std)  : {results[best_clf_name]['mean']:.3f} ± {results[best_clf_name]['std']:.3f}")
print(f"Seuil retenu       : {DECISION_THRESHOLD:.3f}")

Classifieur retenu : Random Forest
AUC train (full)   : 1.000
AUC CV (mean±std)  : 0.871 ± 0.030
Seuil retenu       : 0.499


## 10. Évaluation sur le jeu de test

Le test set contient des images **non augmentées, non vues à l'entraînement**.

In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= DECISION_THRESHOLD).astype(int)

test_auc = roc_auc_score(y_test, y_test_proba)
print(f"AUC test : {test_auc:.3f}")
print()
print(classification_report(y_test, y_test_pred, target_names=['HC', 'PD']))

cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['HC', 'PD'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matrice de confusion — test\nAUC={test_auc:.3f}', fontsize=11)
plt.tight_layout()
plt.savefig(DATA_ROOT / 'confusion_matrix_spiral.png', dpi=80)
plt.show()

AUC test : 0.880

              precision    recall  f1-score   support

          HC       0.68      0.87      0.76        15
          PD       0.82      0.60      0.69        15

    accuracy                           0.73        30
   macro avg       0.75      0.73      0.73        30
weighted avg       0.75      0.73      0.73        30



C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_32580\2278137778.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Export de l'artefact


> **Note :** pour utiliser ce modèle dans l'application, modifiez
> `DEFAULT_MODEL_PATH` dans `src/modalities/drawing/predictor.py`.

In [12]:
import joblib

MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    'pipeline': final_pipeline,
    'hog_params': {
        'img_size':        IMG_SIZE,
        'orientations':    HOG_ORIENTATIONS,
        'pixels_per_cell': HOG_PIXELS_CELL,
        'cells_per_block': HOG_CELLS_BLOCK,
    },
    'lbp_params': {
        'radius':   LBP_RADIUS,
        'n_points': LBP_N_POINTS,
        'n_bins':   LBP_N_BINS,
    },
    'preprocessing':  'otsu_binarization',   # clé pour DrawingPredictor
    'feature_type':   'HOG+LBP_on_binary',
    'model':          f'drawing_spiral_hog_lbp_{best_clf_name.lower().replace(" ", "_")}_v2',
    'classifier':     best_clf_name,
    'threshold':      DECISION_THRESHOLD,
    'cv_auc_mean':    float(results[best_clf_name]['mean']),
    'cv_auc_std':     float(results[best_clf_name]['std']),
    'test_auc':       float(test_auc),
    'trained_on':     'kaggle_spiral_dataset_x8aug',
    'aug_factor':     8,
    'domain_shift':   'Otsu binarization + blur/sharpen aug to reduce paper-vs-canvas gap.',
    'note':           'Dataset Kaggle Spiral (PyImageSearch). Otsu binarization + HOG 8x8 + LBP. '
                      'Prototype exploratoire — pas un dispositif médical.',
}

joblib.dump(artifact, MODEL_PATH)
print(f'Modèle sauvegardé : {MODEL_PATH}')

loaded = joblib.load(MODEL_PATH)
assert loaded['hog_params'] == artifact['hog_params']
assert loaded['preprocessing'] == 'otsu_binarization'
print('Vérification OK.')
print(f"Classifieur  : {loaded['classifier']}")
print(f"Prétraitement: {loaded['preprocessing']}")
print(f"AUC CV       : {loaded['cv_auc_mean']:.3f} ± {loaded['cv_auc_std']:.3f}")
print(f"AUC test     : {loaded['test_auc']:.3f}")
print(f"Seuil        : {loaded['threshold']:.3f}")


Modèle sauvegardé : C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\models\drawing_spiral_v2_hog_lbp_pipeline.joblib
Vérification OK.
Classifieur  : Random Forest
Prétraitement: otsu_binarization
AUC CV       : 0.871 ± 0.030
AUC test     : 0.880
Seuil        : 0.499


## 12. Test d'inférence bout en bout

Simulation du flux réel : image PIL → buffer PNG → features → score.

In [13]:
import io as _io

for label_name, src_imgs, src_labs in [
    ('HC', [img for img, l in zip(test_imgs, test_labs) if l == 0][:3], [0]*3),
    ('PD', [img for img, l in zip(test_imgs, test_labs) if l == 1][:3], [1]*3),
]:
    for img, true_lab in zip(src_imgs, src_labs):
        # Simulation du round-trip navigateur → PNG → PIL
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        received = Image.open(_io.BytesIO(buf.getvalue()))

        feats = extract_features(received).reshape(1, -1)
        score = loaded['pipeline'].predict_proba(feats)[0, 1]
        pred  = 'PD' if score >= loaded['threshold'] else 'HC'
        status = 'OK' if pred == label_name else 'ERREUR'
        print(f'[{status}] Vraie={label_name}  Score PD={score:.3f}  Prédit={pred}')

[OK] Vraie=HC  Score PD=0.400  Prédit=HC

c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(



[ERREUR] Vraie=HC  Score PD=0.558  Prédit=PD


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differe

[OK] Vraie=HC  Score PD=0.453  Prédit=HC
[ERREUR] Vraie=PD  Score PD=0.412  Prédit=HC
[OK] Vraie=PD  Score PD=0.708  Prédit=PD
[OK] Vraie=PD  Score PD=0.707  Prédit=PD


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
